# 🤖 Notebook 02 — H10 Injection Data Scraper (Texas RRC)

**Project:** Texas Injection Wells — Delaware Basin Analysis  
**Author:** Juan David Antolinez  
**Purpose:** Scrape monthly injection data (volume, pressure) for each well in the Delaware Basin from the Texas Railroad Commission (RRC) H10 portal using Selenium and BeautifulSoup.

---

### How It Works
The Texas RRC does not provide a bulk API for H10 injection records. Data is only accessible through a web form, one well at a time. This scraper automates that process:

1. Load the UIC number list generated in Notebook 01
2. For each UIC, open the RRC H10 search form in Chrome (automated)
3. Submit the form and navigate to each Due Date record
4. Extract well header, technical specs, and monthly injection table
5. Apply incremental update logic — skip months already collected, update empty records
6. Save results to Excel with checkpoint every 100 wells

### Incremental Update Logic
When re-running the scraper on an existing dataset:
- **New month** → append row
- **Existing month with BBLS** → skip (already complete)
- **Existing month without BBLS** → replace row (RRC may have updated it)

### Output
- `data/raw/resultados_h10.xlsx` — raw monthly injection data (~233k rows)

> ⚠️ **Runtime:** ~8 hours for 2,875 wells. Run overnight.

## 1. Library Imports

In [ ]:
# ─── STANDARD LIBRARY ─────────────────────────────────────────────────────────
import os
import time
import signal
import sys
from pathlib import Path

# ─── DATA MANIPULATION ────────────────────────────────────────────────────────
import pandas as pd

# ─── WEB SCRAPING ─────────────────────────────────────────────────────────────
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# ─── CONFIGURATION ────────────────────────────────────────────────────────────
BASE_URL = "https://webapps.rrc.texas.gov/H10/searchH10.do"  # RRC H10 portal

## 2. Project Paths & UIC List

Load the list of UIC numbers from the well inventory generated in Notebook 01.

In [ ]:
BASE_DIR  = Path.cwd().parent
WELLS_CSV = BASE_DIR / "data" / "raw" / "wells_delaware_coordinates.csv"

# Load well list and extract unique UIC numbers
delaware_wells = pd.read_csv(WELLS_CSV)
uic_list = (
    delaware_wells['uic_number']
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

print(f"Total UIC numbers to scrape: {len(uic_list):,}")

## 3. Scraper Configuration

| Parameter | Value | Description |
|---|---|---|
| `UIC_NUMBERS` | `uic_list` | Full list of wells to scrape |
| `OUTPUT_FILE` | `data/raw/resultados_h10.xlsx` | Output file path |
| `DELAY` | `0.5s` | Wait time between requests |
| `HEADLESS` | `False` | Show Chrome window while scraping |

## 4. Scraper Functions

The scraper is organized into 5 functions:

| Function | Description |
|---|---|
| `crear_driver()` | Initialize Chrome WebDriver with options |
| `buscar_uic()` | Search a UIC number and extract all Due Date records |
| `extraer_campos()` | Parse well header fields (operator, API, county, etc.) |
| `extraer_campos_footer()` | Parse technical specs (intervals, fluid type, etc.) |
| `extraer_tabla_mensual()` | Parse monthly injection table (BBLS, PSIG, MCF) |
| `main()` | Orchestrate the full scraping loop with checkpointing |

In [ ]:
# ── SCRAPER SETTINGS ──────────────────────────────────────────────────────────
UIC_NUMBERS = uic_list                                            # scrape all wells
# UIC_NUMBERS = uic_list[:5]                                      # test with 5 wells
OUTPUT_FILE = str(BASE_DIR / "data" / "raw" / "resultados_h10.xlsx")
DELAY       = 0.5    # seconds between page requests
HEADLESS    = False  # set True to run Chrome invisibly

# ── DRIVER ────────────────────────────────────────────────────────────────────

def crear_driver():
    """Initialize Chrome WebDriver with anti-detection options."""
    options = Options()
    if HEADLESS:
        options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    driver = webdriver.Chrome(options=options)
    driver.implicitly_wait(1)
    return driver

# ── MAIN SEARCH ───────────────────────────────────────────────────────────────

def buscar_uic(driver, uic_number):
    """Search a UIC number on the RRC H10 portal and return all monthly records."""
    wait = WebDriverWait(driver, 10)

    # Load search form
    driver.get(BASE_URL)

    # Enter UIC number
    campo = wait.until(
        EC.presence_of_element_located(
            (By.NAME, "searchRcrd.uicNoHndlr.inputValue")
        )
    )
    campo.clear()
    campo.send_keys(uic_number)

    # Submit search
    driver.find_element(
        By.CSS_SELECTOR, "input[name='mode'][value='Search']"
    ).click()
    time.sleep(DELAY)

    # Check results
    if "H10 Records Found" not in driver.page_source:
        print(f"  ⚠️  No results for UIC {uic_number}")
        return []

    # Switch to View All
    try:
        select_elem = wait.until(
            EC.presence_of_element_located((By.NAME, "pager.pageSize"))
        )
        Select(select_elem).select_by_value("-1")
        wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "table.DataGrid tr td"))
        )
    except NoSuchElementException:
        pass

    # Collect all Due Date links
    tabla = driver.find_element(By.CSS_SELECTOR, "table.DataGrid")
    filas = tabla.find_elements(By.TAG_NAME, "tr")[1:]

    due_date_links = []
    for fila in filas:
        celdas = fila.find_elements(By.TAG_NAME, "td")
        if len(celdas) >= 1:
            try:
                link = celdas[0].find_element(By.TAG_NAME, "a")
                due_date_links.append({
                    "due_date" : link.text.strip(),
                    "href"     : link.get_attribute("href")
                })
            except NoSuchElementException:
                pass

    print(f"   → {len(due_date_links)} Due Dates found")

    # Visit each Due Date and extract data
    resultados = []
    for j, item in enumerate(due_date_links, 1):
        print(f"     [{j}/{len(due_date_links)}] {item['due_date']} ...", end=" ")
        try:
            driver.get(item["href"])
            time.sleep(DELAY)

            campos         = extraer_campos(driver)
            campos_footer  = extraer_campos_footer(driver)
            filas_mensuales = extraer_tabla_mensual(driver)

            print(f"→ {campos.get('Operator Name', '?')} ({len(filas_mensuales)} months)")

            for fila_mes in filas_mensuales:
                resultados.append({
                    "UIC Number" : uic_number,
                    "Due Date"   : item["due_date"],
                    **campos,
                    **campos_footer,
                    **fila_mes,
                })

            if not filas_mensuales:
                resultados.append({
                    "UIC Number" : uic_number,
                    "Due Date"   : item["due_date"],
                    **campos,
                    **campos_footer,
                })

        except Exception as e:
            print(f"→ ERROR: {e}")
            resultados.append({
                "UIC Number"    : uic_number,
                "Due Date"      : item["due_date"],
                "Operator Name" : "ERROR",
            })

    return resultados

# ── WELL HEADER FIELDS ────────────────────────────────────────────────────────

def extraer_campos(driver):
    """Extract well header fields: operator, API number, county, etc."""
    soup = BeautifulSoup(driver.page_source, "html.parser")

    campos = {
        "Operator Name" : "1. CURRENT OPERATOR NAME",
        "District"      : "3. RRC DISTRICT NO.",
        "API No."       : "5. API NO.",
        "Oil Lease No." : "6. OIL LEASE NO.",
        "Field Name"    : "7. FIELD NAME",
        "Lease Name"    : "9. LEASE NAME",
        "County"        : "10. COUNTY",
        "Well No."      : "11. WELL NO.",
    }

    resultado = {}
    all_tds = soup.find_all("td")

    for nombre, label in campos.items():
        resultado[nombre] = "NOT FOUND"
        for td in all_tds:
            tds_internos = td.find_all("td")
            if label in td.get_text() and len(tds_internos) == 2:
                resultado[nombre] = tds_internos[1].get_text(strip=True)
                break

    return resultado

# ── WELL FOOTER FIELDS ────────────────────────────────────────────────────────

def extraer_campos_footer(driver):
    """Extract technical specs: injection intervals, fluid type, tubing packer depth."""
    soup = BeautifulSoup(driver.page_source, "html.parser")

    tabla = soup.find("table", {"summary": "Use the data entry fields in this table to enter injection data."})
    if not tabla:
        return {
            "16. Interval From (ft)" : "",
            "16. Interval To (ft)"   : "",
            "17. Tubing Packer (ft)" : "",
            "18. Fluids From Others" : "",
            "19. Injection Through"  : "",
            "20. Fluid Type"         : "",
        }

    tds = tabla.find_all("td")
    strongs_16 = [s.get_text(strip=True) for s in tds[0].find_all("strong") if s.get_text(strip=True) and s.get_text(strip=True) != "."]

    return {
        "16. Interval From (ft)" : strongs_16[0] if len(strongs_16) > 0 else "",
        "16. Interval To (ft)"   : strongs_16[1] if len(strongs_16) > 1 else "",
        "17. Tubing Packer (ft)" : tds[1].find("strong").get_text(strip=True),
        "18. Fluids From Others" : tds[2].find("strong").get_text(strip=True),
        "19. Injection Through"  : tds[3].find("strong").get_text(strip=True),
        "20. Fluid Type"         : tds[4].find("strong").get_text(strip=True),
    }

# ── MONTHLY INJECTION TABLE ───────────────────────────────────────────────────

def extraer_tabla_mensual(driver):
    """
    Extract monthly injection records.
    Columns: Month/Yr | Avg PSIG | Max PSIG | BBLS | MCF | # Readings | Min PSIG AP | Max PSIG AP
    """
    soup = BeautifulSoup(driver.page_source, "html.parser")

    tabla = soup.find("table", {"summary": "Use this table to enter monthly pressures and volumes."})
    if not tabla:
        return []

    filas_datos = []
    for fila in tabla.find_all("tr"):
        celdas = fila.find_all("td")
        if len(celdas) == 8:  # data rows have exactly 8 columns
            filas_datos.append({
                "Month/Yr"    : celdas[0].get_text(strip=True),
                "Avg PSIG"    : celdas[1].get_text(strip=True),
                "Max PSIG"    : celdas[2].get_text(strip=True),
                "BBLS"        : celdas[3].get_text(strip=True),
                "MCF"         : celdas[4].get_text(strip=True),
                "# Readings"  : celdas[5].get_text(strip=True),
                "Min PSIG AP" : celdas[6].get_text(strip=True),
                "Max PSIG AP" : celdas[7].get_text(strip=True),
            })

    return filas_datos

# ── MAIN LOOP ─────────────────────────────────────────────────────────────────

def main():
    """Main scraping loop with incremental update logic and checkpoint saving."""

    def save_and_exit(sig, frame):
        """Handle Ctrl+C — save progress before exiting."""
        print("\n⚠️  Interrupt detected — saving checkpoint...")
        if todos_resultados:
            try:
                pd.DataFrame(todos_resultados).to_excel(OUTPUT_FILE, index=False)
                print(f"✅  Saved: {OUTPUT_FILE} ({len(todos_resultados)} rows)")
            except Exception:
                pd.DataFrame(todos_resultados).to_csv(OUTPUT_FILE.replace(".xlsx", ".csv"), index=False)
                print("✅  CSV backup saved")
        sys.exit(0)

    signal.signal(signal.SIGINT, save_and_exit)

    print("🛢️  RRC Texas H10 Scraper")
    print(f"   Wells to scrape : {len(UIC_NUMBERS):,}")
    print(f"   Output file     : {OUTPUT_FILE}\n")

    # Load existing data if available (incremental mode)
    try:
        df_existente     = pd.read_excel(OUTPUT_FILE, dtype=str)
        uics_procesados  = set(df_existente["UIC Number"].astype(str).tolist())
        todos_resultados = df_existente.to_dict("records")
        print(f"   ♻️  Resuming — {len(uics_procesados):,} UICs already in database\n")
    except FileNotFoundError:
        uics_procesados  = set()
        todos_resultados = []
        df_existente     = pd.DataFrame()
        print("   🆕  No existing file — starting fresh\n")

    driver = crear_driver()

    try:
        for i, uic in enumerate(UIC_NUMBERS, 1):
            print(f"[{i}/{len(UIC_NUMBERS)}] UIC: {uic}")

            try:
                filas = buscar_uic(driver, uic)

                for fila in filas:
                    mes = fila.get('Month/Yr')

                    # First run — no existing data
                    if df_existente.empty:
                        todos_resultados.append(fila)
                        continue

                    # Check if this UIC + month already exists
                    mask = (
                        (df_existente['UIC Number'].astype(str) == str(uic)) &
                        (df_existente['Month/Yr'] == mes)
                    )

                    existe     = mask.any()
                    tiene_bbls = False

                    if existe:
                        bbls_val   = df_existente.loc[mask, 'BBLS'].values[0]
                        tiene_bbls = pd.notna(bbls_val) and str(bbls_val).strip() not in ['', 'nan']

                    if not existe:
                        # Case 1 — new month, append
                        todos_resultados.append(fila)
                    elif existe and not tiene_bbls:
                        # Case 2 — month exists but BBLS is empty, replace
                        df_existente     = df_existente[~mask]
                        todos_resultados = [
                            r for r in todos_resultados
                            if not (str(r.get('UIC Number')) == str(uic) and r.get('Month/Yr') == mes)
                        ]
                        todos_resultados.append(fila)
                    else:
                        # Case 3 — month exists with BBLS, skip
                        pass

                print(f"   ✅ {len(filas)} rows processed\n")

            except TimeoutException:
                print(f"   ❌ Timeout\n")
            except Exception as e:
                print(f"   ❌ Error: {e}\n")

            # Save checkpoint every 100 UICs
            if i % 100 == 0:
                print(f"   💾 Saving checkpoint — {i} UICs processed...")
                try:
                    pd.DataFrame(todos_resultados).to_excel(OUTPUT_FILE, index=False)
                    print("   ✅ Checkpoint saved")
                except Exception as e:
                    print(f"⚠️  Excel failed: {e} — saving CSV backup...")
                    pd.DataFrame(todos_resultados).to_csv(OUTPUT_FILE.replace(".xlsx", ".csv"), index=False)

            if i < len(UIC_NUMBERS):
                time.sleep(DELAY)

    finally:
        driver.quit()

    # Save final output
    if todos_resultados:
        df = pd.DataFrame(todos_resultados)
        df = df.where(df.notna(), other='')
        df = df.astype(str).replace('nan', '')
        try:
            df.to_excel(OUTPUT_FILE, index=False)
            print(f"\n✅  Saved: {OUTPUT_FILE} ({len(df):,} rows)")
        except Exception as e:
            df.to_csv(OUTPUT_FILE.replace(".xlsx", ".csv"), index=False)
            print(f"✅  CSV saved: {OUTPUT_FILE.replace('.xlsx', '.csv')} ({len(df):,} rows)")
        return df
    else:
        print("⚠️  No results obtained.")
        return None


if __name__ == "__main__":
    df = main()

🛢️  RRC Texas H10 Scraper
   Pozos a consultar : 11
   Output            : g:\My Drive\Portafolio analisis de datos\texas-injection-wells\data\raw\resultados_h10.xlsx

   ♻️  Retomando — 2349 UICs ya procesados

[1/11] UIC: 117332
   → 8 Due Dates encontrados
     [1/8] 06/2026 ... → COG OPERATING LLC (12 meses)
     [2/8] 06/2025 ... → COG OPERATING LLC (12 meses)
     [3/8] 06/2024 ... → COG OPERATING LLC (12 meses)
     [4/8] 06/2023 ... → COG OPERATING LLC (12 meses)
     [5/8] 06/2022 ... → COG OPERATING LLC (12 meses)
     [6/8] 06/2021 ... → COG OPERATING LLC (12 meses)
     [7/8] 06/2020 ... → COG OPERATING LLC (12 meses)
     [8/8] 06/2019 ... → COG OPERATING LLC (12 meses)
   ✅ 96 filas obtenidas

[2/11] UIC: 117339
   → 8 Due Dates encontrados
     [1/8] 06/2026 ... → PDEH LLC (12 meses)
     [2/8] 06/2025 ... → PDEH LLC (12 meses)
     [3/8] 06/2024 ... → PDEH LLC (12 meses)
     [4/8] 06/2023 ... → PDEH LLC (12 meses)
     [5/8] 06/2022 ... → PDEH LLC (12 meses)
     [6/8]